# Check Domain Coverage in WDC October 2024 Datasets

Checks if specific domains exist in the **Web Data Commons (WDC) October 2024** structured data extractions  
(RDFa, Microdata, Embedded JSON-LD, Microformats) **without downloading the full nquads files**.

## Strategy

1. **WDC HTML stats pages** — WDC publishes per-domain rankings (top 1,000 by triples) as HTML pages.  
   These are lightweight — we scrape them with `requests` + `BeautifulSoup`.
2. **Common Crawl CDX API** — Gives URL counts per host from the underlying CC-MAIN-2024-42 crawl.  
   Useful for domains not in the WDC top 1,000 (confirms they were crawled, even if low JSON-LD coverage).
3. **Streaming nquads** — Optional spot-check: stream the first N lines of a shard to verify domain presence.

> **Note on granularity:** WDC stats are at **pay-level domain (PLD)** level (e.g. `wikipedia.org`).  
> `en.wikipedia.org` will appear as `wikipedia.org` in the rankings.

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
# Uncomment and run once if needed:
# !pip install requests beautifulsoup4 pandas lxml

In [4]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import requests
import pandas as pd
import time
import re
import gzip
from collections import defaultdict
from urllib.parse import urlparse
from bs4 import BeautifulSoup

TARGET_DOMAINS = [
    "en.wikipedia.org",
    "france24.com",
    "bbc.com",
    "lemonde.fr",
    "cnn.com",
    "aljazeera.com",
    "thetimes.com",
    "politifact.com",
]

# Pay-level domain (eTLD+1) for each target:
# en.wikipedia.org → wikipedia.org  (WDC stats level)
def get_pld(host):
    parts = host.split(".")
    return ".".join(parts[-2:]) if len(parts) >= 2 else host

pld_to_targets = defaultdict(list)
for d in TARGET_DOMAINS:
    pld_to_targets[get_pld(d)].append(d)

# PLD set to search in stats tables
TARGET_PLDS = set(pld_to_targets.keys())

print("Target domains:", TARGET_DOMAINS)
print("\nPLD-level lookup (used for WDC stats):")
for pld, hosts in pld_to_targets.items():
    print(f"  {pld}  ←  {hosts}")

Target domains: ['en.wikipedia.org', 'france24.com', 'bbc.com', 'lemonde.fr', 'cnn.com', 'aljazeera.com', 'thetimes.com', 'politifact.com']

PLD-level lookup (used for WDC stats):
  wikipedia.org  ←  ['en.wikipedia.org']
  france24.com  ←  ['france24.com']
  bbc.com  ←  ['bbc.com']
  lemonde.fr  ←  ['lemonde.fr']
  cnn.com  ←  ['cnn.com']
  aljazeera.com  ←  ['aljazeera.com']
  thetimes.com  ←  ['thetimes.com']
  politifact.com  ←  ['politifact.com']


## Part 1 — WDC HTML Stats Pages (Top 1,000 domains per format)

WDC publishes lightweight HTML pages with the top 1,000 domains by extracted triples for each format.  
We scrape them all and search for our target domains.  
No authentication, no huge downloads — each page is a small HTML table.

In [5]:
# ── WDC stats page URLs ────────────────────────────────────────────────────────
WDC_STATS_BASE = "http://webdatacommons.org/structureddata/2024-12/stats/"

# Each format has a "More" page with top 1,000 domains by extracted triples.
FORMAT_PAGES = {
    "JSON-LD":      "top_domains_by_extracted_triples_for_extractor_html-embedded-jsonld.html",
    "Microdata":    "top_domains_by_extracted_triples_for_extractor_html-microdata.html",
    "RDFa":         "top_domains_by_extracted_triples_for_extractor_html-rdfa.html",
    "MF-hCard":     "top_domains_by_extracted_triples_for_extractor_html-mf-hcard.html",
    "MF-XFN":       "top_domains_by_extracted_triples_for_extractor_html-mf-xfn.html",
    "MF-adr":       "top_domains_by_extracted_triples_for_extractor_html-mf-adr.html",
    "MF-geo":       "top_domains_by_extracted_triples_for_extractor_html-mf-geo.html",
    "MF-hCalendar": "top_domains_by_extracted_triples_for_extractor_html-mf-hcalendar.html",
    "MF-hReview":   "top_domains_by_extracted_triples_for_extractor_html-mf-hreview.html",
    "MF-hListing":  "top_domains_by_extracted_triples_for_extractor_html-mf-hlisting.html",
    "MF-hRecipe":   "top_domains_by_extracted_triples_for_extractor_html-mf-hrecipe.html",
    "MF-hResume":   "top_domains_by_extracted_triples_for_extractor_html-mf-hresume.html",
    "MF-species":   "top_domains_by_extracted_triples_for_extractor_html-mf-species.html",
}


def scrape_domain_table(url):
    """
    Fetch a WDC stats HTML page and extract the domain ranking list.

    The pages do NOT use <table> — they use a numbered list of anchor tags
    with triple counts in parentheses, e.g.:
        <li><a href="...">blogspot.com</a> (30,926,064)</li>
    or inline text like:
        1. blogspot.com (30,926,064)

    Strategy:
    1. Try <table> (fallback for older pages).
    2. Try <ol>/<ul> list items containing <a> tags.
    3. Regex over the raw HTML for the pattern: domain + (number).
    Returns a DataFrame with columns: rank, domain, triples.
    """
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
    except Exception as e:
        print(f"  ERROR fetching {url}: {e}")
        return pd.DataFrame()

    html  = resp.text
    soup  = BeautifulSoup(html, "lxml")
    rows  = []

    # ── Strategy 1: <table> ────────────────────────────────────────────────────
    table = soup.find("table")
    if table:
        for tr in table.find_all("tr"):
            cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
            if cells:
                rows.append(cells)
        if len(rows) > 1:
            header = rows[0]
            data   = rows[1:]
            df = pd.DataFrame(data, columns=header[:len(data[0])] if data else header)
            col_lower = {c: c.lower() for c in df.columns}
            domain_col = next((c for c, cl in col_lower.items() if any(k in cl for k in ('domain','host','pld'))), df.columns[0])
            triple_col = next((c for c, cl in col_lower.items() if any(k in cl for k in ('triple','count'))), df.columns[1] if len(df.columns) > 1 else None)
            df = df.rename(columns={domain_col: "domain"})
            if triple_col:
                df = df.rename(columns={triple_col: "triples"})
                df["triples"] = pd.to_numeric(df["triples"].astype(str).str.replace(r"[,\s]", "", regex=True), errors="coerce")
            df.insert(0, "rank", range(1, len(df) + 1))
            df["domain"] = df["domain"].astype(str).str.strip()
            return df

    # ── Strategy 2: <li> items with <a> links ─────────────────────────────────
    # Pattern: <a href="...">domain.tld</a> followed by (N,NNN,NNN)
    NUM_RE = re.compile(r'\(\s*([\d,]+)\s*\)')
    list_items = soup.find_all("li")
    for item in list_items:
        a = item.find("a")
        if a:
            domain_text = a.get_text(strip=True)
            rest = item.get_text(separator=" ")
            m = NUM_RE.search(rest)
            triples = int(m.group(1).replace(",", "")) if m else None
            rows.append([domain_text, triples])

    if rows:
        df = pd.DataFrame(rows, columns=["domain", "triples"])
        df["domain"]  = df["domain"].astype(str).str.strip()
        df["triples"] = pd.to_numeric(df["triples"], errors="coerce")
        df.insert(0, "rank", range(1, len(df) + 1))
        return df

    # ── Strategy 3: regex over raw HTML ───────────────────────────────────────
    # Matches lines like: blogspot.com (30,926,064)  or  1. bbc.com (123,456)
    INLINE_RE = re.compile(
        r'(?:^\s*\d+\.\s+)?'           # optional leading rank "1. "
        r'([\w][\w\-]*(?:\.[\w\-]+)+)' # domain (at least one dot)
        r'\s*\(\s*([\d,]+)\s*\)',       # (triple_count)
        re.MULTILINE
    )
    for i, m in enumerate(INLINE_RE.finditer(html), start=1):
        domain  = m.group(1).strip()
        triples = int(m.group(2).replace(",", ""))
        rows.append([i, domain, triples])

    if rows:
        df = pd.DataFrame(rows, columns=["rank", "domain", "triples"])
        df["domain"] = df["domain"].astype(str).str.strip()
        return df

    print(f"  Could not parse any domain rows from {url}")
    return pd.DataFrame()


# ── Fetch all format pages and filter for target domains ───────────────────────
results = []

for fmt_name, page_file in FORMAT_PAGES.items():
    url = WDC_STATS_BASE + page_file
    print(f"Fetching {fmt_name} ... ", end="", flush=True)
    df = scrape_domain_table(url)
    if df.empty:
        print("(empty)")
        continue
    print(f"{len(df)} rows")

    matched = df[df["domain"].isin(TARGET_PLDS)].copy()
    if not matched.empty:
        matched.insert(0, "format", fmt_name)
        results.append(matched)
    time.sleep(0.3)


Fetching JSON-LD ... 1000 rows
Fetching Microdata ... 1000 rows
Fetching RDFa ... 1000 rows
Fetching MF-hCard ... 1000 rows
Fetching MF-XFN ... 1000 rows
Fetching MF-adr ... 1000 rows
Fetching MF-geo ... 1000 rows
Fetching MF-hCalendar ... 1000 rows
Fetching MF-hReview ... 1000 rows
Fetching MF-hListing ... 1000 rows
Fetching MF-hRecipe ... 1000 rows
Fetching MF-hResume ... 61 rows
Fetching MF-species ... 291 rows


In [6]:
# ── Display results ────────────────────────────────────────────────────────────
if results:
    combined = pd.concat(results, ignore_index=True)

    print("\n" + "="*70)
    print("FOUND — Target domains in WDC October 2024 (top 1,000 per format)")
    print("="*70)
    # Show all columns available
    print(combined.to_string(index=False))

    print("\n" + "="*70)
    print("JSON-LD only:")
    print("="*70)
    jsonld = combined[combined["format"] == "JSON-LD"]
    print(jsonld.to_string(index=False) if not jsonld.empty else "  None found in top 1,000")
else:
    print("No matches found in any format's top 1,000.")

# Report which domains were NOT found at all
if results:
    found_plds = set(combined["domain"].unique())
else:
    found_plds = set()

missing_plds = TARGET_PLDS - found_plds
if missing_plds:
    print("\n" + "="*70)
    print("NOT in top 1,000 for any format:")
    for pld in sorted(missing_plds):
        print(f"  {pld}  (requested: {pld_to_targets[pld]})")
    print("→ These may still be in the dataset but below the top-1,000 threshold.")
    print("  Use Part 2 (CC CDX API) and Part 3 (streaming) to investigate further.")


FOUND — Target domains in WDC October 2024 (top 1,000 per format)
      format  rank         domain  triples
     JSON-LD     5  wikipedia.org      NaN
     JSON-LD   158        cnn.com      NaN
    MF-hCard     2  wikipedia.org      NaN
      MF-adr     6  wikipedia.org      NaN
      MF-adr    14 politifact.com      NaN
      MF-geo     1  wikipedia.org      NaN
MF-hCalendar     1  wikipedia.org      NaN
  MF-hRecipe    49  wikipedia.org      NaN
  MF-species     1  wikipedia.org      NaN

JSON-LD only:
 format  rank        domain  triples
JSON-LD     5 wikipedia.org      NaN
JSON-LD   158       cnn.com      NaN

NOT in top 1,000 for any format:
  aljazeera.com  (requested: ['aljazeera.com'])
  bbc.com  (requested: ['bbc.com'])
  france24.com  (requested: ['france24.com'])
  lemonde.fr  (requested: ['lemonde.fr'])
  thetimes.com  (requested: ['thetimes.com'])
→ These may still be in the dataset but below the top-1,000 threshold.
  Use Part 2 (CC CDX API) and Part 3 (streaming) to in